# 实验 C：GoEmotions 28 类多标签情绪差异评估

固定模型：

- `SamLowe/roberta-base-go_emotions`

固定数据：

- `google-research-datasets/go_emotions`
- `simplified` 配置
- 27 种细粒度情绪 + `neutral`
- 每条文本可拥有一个或多个标签

本实验不会把多标签任务错误地简化成普通单标签分类，而是计算：

1. Micro / Macro / Weighted / Samples F1
2. Exact-match accuracy
3. Hamming loss
4. 每个标签的 Precision、Recall、F1、PR-AUC 和样本量
5. 固定阈值 0.5、验证集优化的全局阈值、验证集优化的逐标签阈值
6. 标签样本量与 F1 的关系
7. Neutral 专项指标
8. 高置信度 False Positive 和 False Negative

> 模型本身是在 GoEmotions 上训练的，因此这里是同源官方测试集评估。阈值只能在验证集上选择，不能在测试集上调参。

In [ ]:
!pip -q install -U transformers datasets accelerate scikit-learn scipy pandas==2.2.2 matplotlib

In [ ]:
import random

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from datasets import load_dataset
from IPython.display import display
from scipy.stats import spearmanr
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    hamming_loss,
    jaccard_score,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
)
from transformers import AutoModelForSequenceClassification, AutoTokenizer

SEED = 42
MODEL_NAME = "SamLowe/roberta-base-go_emotions"
DATASET_NAME = "google-research-datasets/go_emotions"
DATASET_CONFIG = "simplified"

BATCH_SIZE = 64
MAX_LENGTH = 128

# 阈值搜索网格。需要更精细时可改成 np.arange(0.01, 1.00, 0.01)。
THRESHOLD_GRID = np.arange(0.05, 0.96, 0.05)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("运行设备：", device)

## 1. 加载 GoEmotions

验证集用于选择阈值；测试集只用于最终评估。

In [ ]:
dataset = load_dataset(DATASET_NAME, DATASET_CONFIG)
validation_ds = dataset["validation"]
test_ds = dataset["test"]

label_feature = test_ds.features["labels"]
if hasattr(label_feature, "feature") and hasattr(label_feature.feature, "names"):
    dataset_label_names = list(label_feature.feature.names)
elif hasattr(label_feature, "names"):
    dataset_label_names = list(label_feature.names)
else:
    raise TypeError(f"无法从标签特征读取名称：{label_feature}")

n_labels = len(dataset_label_names)

print("验证集样本数：", len(validation_ds))
print("测试集样本数：", len(test_ds))
print("标签数：", n_labels)
print(dataset_label_names)
display(pd.DataFrame({
    "text": test_ds["text"][:5],
    "labels": [
        [dataset_label_names[i] for i in ids]
        for ids in test_ds["labels"][:5]
    ],
}))

## 2. 加载模型并对齐标签顺序

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)
model.eval()

def normalize_label(value):
    return str(value).strip().lower().replace(" ", "_")

model_id2label = {
    int(k): normalize_label(v)
    for k, v in model.config.id2label.items()
}
dataset_label_names = [normalize_label(x) for x in dataset_label_names]

if set(model_id2label.values()) != set(dataset_label_names):
    raise ValueError(
        "模型与数据集标签集合不一致。\n"
        f"模型独有：{sorted(set(model_id2label.values()) - set(dataset_label_names))}\n"
        f"数据集独有：{sorted(set(dataset_label_names) - set(model_id2label.values()))}"
    )

# dataset_index -> model_logit_index
model_label_to_id = {name: idx for idx, name in model_id2label.items()}
dataset_to_model_indices = [
    model_label_to_id[name] for name in dataset_label_names
]

print("标签已按名称安全对齐。")

## 3. 构造多标签真实矩阵并批量推理

模型输出的每一个维度都经过 Sigmoid，得到该标签独立出现的概率。

In [ ]:
def labels_to_multihot(label_lists, num_labels):
    matrix = np.zeros((len(label_lists), num_labels), dtype=np.int8)
    for row_index, label_ids in enumerate(label_lists):
        matrix[row_index, list(label_ids)] = 1
    return matrix

def predict_probabilities(texts):
    probability_batches = []

    for start in range(0, len(texts), BATCH_SIZE):
        batch_texts = texts[start : start + BATCH_SIZE]
        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        ).to(device)

        with torch.inference_mode():
            logits = model(**encoded).logits
            probabilities = torch.sigmoid(logits)

        # 转成数据集标签顺序。
        probabilities = probabilities[:, dataset_to_model_indices]
        probability_batches.append(probabilities.cpu().numpy())

    return np.vstack(probability_batches)

y_val_true = labels_to_multihot(validation_ds["labels"], n_labels)
y_test_true = labels_to_multihot(test_ds["labels"], n_labels)

val_prob = predict_probabilities(validation_ds["text"])
test_prob = predict_probabilities(test_ds["text"])

print("验证集概率矩阵：", val_prob.shape)
print("测试集概率矩阵：", test_prob.shape)

## 4. 在验证集选择阈值

比较三种策略：

1. **固定阈值 0.5**
2. **全局阈值**：所有标签使用同一个阈值，在验证集上最大化 Macro-F1
3. **逐标签阈值**：每个标签单独在验证集上最大化该标签 F1

逐标签阈值通常能提高低频标签 Recall，但可能降低 Precision。  
阈值选择全程不查看测试集标签。

In [ ]:
def find_best_global_threshold(y_true, probabilities, grid):
    records = []
    for threshold in grid:
        predictions = (probabilities >= threshold).astype(np.int8)
        records.append({
            "threshold": float(threshold),
            "macro_f1": f1_score(
                y_true, predictions, average="macro", zero_division=0
            ),
            "micro_f1": f1_score(
                y_true, predictions, average="micro", zero_division=0
            ),
        })
    table = pd.DataFrame(records)
    best_row = table.sort_values(
        ["macro_f1", "micro_f1", "threshold"],
        ascending=[False, False, False],
    ).iloc[0]
    return float(best_row["threshold"]), table

def find_best_per_label_thresholds(y_true, probabilities, grid):
    thresholds = np.zeros(y_true.shape[1], dtype=float)
    best_f1_values = np.zeros(y_true.shape[1], dtype=float)

    for label_index in range(y_true.shape[1]):
        candidates = []
        for threshold in grid:
            predictions = (
                probabilities[:, label_index] >= threshold
            ).astype(np.int8)
            score = f1_score(
                y_true[:, label_index],
                predictions,
                zero_division=0,
            )
            candidates.append((score, float(threshold)))

        # F1 相同则选择更高阈值，以减少不必要的 False Positive。
        best_score, best_threshold = max(
            candidates, key=lambda item: (item[0], item[1])
        )
        thresholds[label_index] = best_threshold
        best_f1_values[label_index] = best_score

    return thresholds, best_f1_values

best_global_threshold, global_threshold_search_df = (
    find_best_global_threshold(y_val_true, val_prob, THRESHOLD_GRID)
)
per_label_thresholds, val_best_per_label_f1 = (
    find_best_per_label_thresholds(y_val_true, val_prob, THRESHOLD_GRID)
)

threshold_table = pd.DataFrame({
    "emotion": dataset_label_names,
    "per_label_threshold": per_label_thresholds,
    "validation_best_f1": val_best_per_label_f1,
})

print("验证集最佳全局阈值：", best_global_threshold)
display(global_threshold_search_df.sort_values("threshold"))
display(threshold_table)

## 5. 三种阈值策略的测试集总体指标

- **Exact-match accuracy**：一条文本的全部 28 个标签必须完全正确。
- **Hamming loss**：所有“样本 × 标签”二元判断中的错误比例，越低越好。
- **Samples-F1**：先计算每条文本的 F1，再求平均。
- **Label cardinality**：平均每条文本被预测多少个标签。
- **No-label rate**：没有任何预测标签的文本比例。

In [ ]:
def apply_thresholds(probabilities, thresholds):
    thresholds = np.asarray(thresholds)
    return (probabilities >= thresholds).astype(np.int8)

def multilabel_summary(y_true, predictions, strategy_name):
    return {
        "strategy": strategy_name,
        "exact_match_accuracy": accuracy_score(y_true, predictions),
        "hamming_loss": hamming_loss(y_true, predictions),
        "micro_precision": precision_score(
            y_true, predictions, average="micro", zero_division=0
        ),
        "micro_recall": recall_score(
            y_true, predictions, average="micro", zero_division=0
        ),
        "micro_f1": f1_score(
            y_true, predictions, average="micro", zero_division=0
        ),
        "macro_f1": f1_score(
            y_true, predictions, average="macro", zero_division=0
        ),
        "weighted_f1": f1_score(
            y_true, predictions, average="weighted", zero_division=0
        ),
        "samples_f1": f1_score(
            y_true, predictions, average="samples", zero_division=0
        ),
        "samples_jaccard": jaccard_score(
            y_true, predictions, average="samples", zero_division=0
        ),
        "true_label_cardinality": y_true.sum(axis=1).mean(),
        "predicted_label_cardinality": predictions.sum(axis=1).mean(),
        "no_label_prediction_rate": (predictions.sum(axis=1) == 0).mean(),
    }

pred_fixed_05 = apply_thresholds(test_prob, 0.5)
pred_global = apply_thresholds(test_prob, best_global_threshold)
pred_per_label = apply_thresholds(test_prob, per_label_thresholds)

strategy_predictions = {
    "fixed_0.50": pred_fixed_05,
    f"global_{best_global_threshold:.2f}": pred_global,
    "per_label_validation_optimized": pred_per_label,
}

strategy_summary_df = pd.DataFrame([
    multilabel_summary(y_test_true, predictions, strategy_name)
    for strategy_name, predictions in strategy_predictions.items()
])

display(strategy_summary_df.style.format({
    "exact_match_accuracy": "{:.2%}",
    "hamming_loss": "{:.4f}",
    "micro_precision": "{:.2%}",
    "micro_recall": "{:.2%}",
    "micro_f1": "{:.2%}",
    "macro_f1": "{:.2%}",
    "weighted_f1": "{:.2%}",
    "samples_f1": "{:.2%}",
    "samples_jaccard": "{:.2%}",
    "true_label_cardinality": "{:.3f}",
    "predicted_label_cardinality": "{:.3f}",
    "no_label_prediction_rate": "{:.2%}",
}))

## 6. 选择主报告策略

默认使用“验证集逐标签优化阈值”。  
研究报告中应同时保留固定 0.5 的结果，便于与模型卡或其他研究比较。

In [ ]:
REPORT_STRATEGY = "per_label_validation_optimized"
y_test_pred = strategy_predictions[REPORT_STRATEGY]
report_thresholds = per_label_thresholds

print("主报告策略：", REPORT_STRATEGY)

## 7. 每个情绪标签的 Precision、Recall、F1、PR-AUC

PR-AUC 对低频标签通常比普通 accuracy 更有解释力。  
one-vs-rest accuracy 仍会输出，但不应作为主要指标。

In [ ]:
def wilson_interval(correct, total, z=1.96):
    if total == 0:
        return np.nan, np.nan
    p = correct / total
    denominator = 1 + z**2 / total
    center = (p + z**2 / (2 * total)) / denominator
    margin = (
        z
        * np.sqrt(p * (1 - p) / total + z**2 / (4 * total**2))
        / denominator
    )
    return max(0.0, center - margin), min(1.0, center + margin)

precision, recall, f1, support = precision_recall_fscore_support(
    y_test_true,
    y_test_pred,
    average=None,
    zero_division=0,
)

rows = []
n_samples = len(y_test_true)

for i, label in enumerate(dataset_label_names):
    true_col = y_test_true[:, i]
    pred_col = y_test_pred[:, i]

    tp = int(((true_col == 1) & (pred_col == 1)).sum())
    fp = int(((true_col == 0) & (pred_col == 1)).sum())
    fn = int(((true_col == 1) & (pred_col == 0)).sum())
    tn = int(((true_col == 0) & (pred_col == 0)).sum())

    ci_low, ci_high = wilson_interval(tp, int(support[i]))
    pr_auc = average_precision_score(true_col, test_prob[:, i])

    rows.append({
        "emotion": label,
        "support": int(support[i]),
        "prevalence": support[i] / n_samples,
        "threshold": report_thresholds[i],
        "precision": precision[i],
        "recall": recall[i],
        "recall_ci95_low": ci_low,
        "recall_ci95_high": ci_high,
        "f1": f1[i],
        "pr_auc": pr_auc,
        "one_vs_rest_accuracy": (tp + tn) / n_samples,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
    })

per_label_df = (
    pd.DataFrame(rows)
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
)

display(per_label_df.style.format({
    "prevalence": "{:.2%}",
    "threshold": "{:.2f}",
    "precision": "{:.2%}",
    "recall": "{:.2%}",
    "recall_ci95_low": "{:.2%}",
    "recall_ci95_high": "{:.2%}",
    "f1": "{:.2%}",
    "pr_auc": "{:.3f}",
    "one_vs_rest_accuracy": "{:.2%}",
}))

## 8. 最佳、最弱标签与样本量关系

In [ ]:
best_label = per_label_df.iloc[0]
worst_label = per_label_df.iloc[-1]
f1_gap = best_label["f1"] - worst_label["f1"]

spearman_result = spearmanr(per_label_df["support"], per_label_df["f1"])
support_f1_df = pd.DataFrame([{
    "spearman_rho_support_vs_f1": spearman_result.statistic,
    "p_value": spearman_result.pvalue,
}])

print(
    f"F1 最佳标签：{best_label['emotion']}，F1={best_label['f1']:.2%}\n"
    f"F1 最弱标签：{worst_label['emotion']}，F1={worst_label['f1']:.2%}\n"
    f"最大 F1 差距：{f1_gap * 100:.2f} 个百分点"
)
display(support_f1_df.style.format({
    "spearman_rho_support_vs_f1": "{:.4f}",
    "p_value": "{:.6g}",
}))

## 9. Neutral 专项指标

GoEmotions 中 `neutral` 是 28 个标签之一。这里分别分析：

- Neutral Precision / Recall / F1
- 真实仅有 neutral 的样本，被预测出非 neutral 标签的比例
- 真实不含 neutral 的样本，被预测出 neutral 的比例

In [ ]:
neutral_index = dataset_label_names.index("neutral")

true_neutral = y_test_true[:, neutral_index] == 1
pred_neutral = y_test_pred[:, neutral_index] == 1
true_neutral_only = true_neutral & (y_test_true.sum(axis=1) == 1)
pred_any_emotion = (
    y_test_pred[:, np.arange(n_labels) != neutral_index].sum(axis=1) > 0
)

neutral_row = per_label_df[per_label_df["emotion"] == "neutral"].iloc[0]

neutral_false_emotional_rate = (
    pred_any_emotion[true_neutral_only].mean()
    if true_neutral_only.any()
    else np.nan
)
non_neutral_mask = ~true_neutral
neutral_false_neutral_rate = (
    pred_neutral[non_neutral_mask].mean()
    if non_neutral_mask.any()
    else np.nan
)

neutral_summary_df = pd.DataFrame([{
    "neutral_support": int(true_neutral.sum()),
    "neutral_only_support": int(true_neutral_only.sum()),
    "neutral_precision": neutral_row["precision"],
    "neutral_recall": neutral_row["recall"],
    "neutral_f1": neutral_row["f1"],
    "neutral_pr_auc": neutral_row["pr_auc"],
    "false_emotional_rate_on_neutral_only": neutral_false_emotional_rate,
    "false_neutral_rate_on_non_neutral": neutral_false_neutral_rate,
}])

display(neutral_summary_df.style.format({
    "neutral_precision": "{:.2%}",
    "neutral_recall": "{:.2%}",
    "neutral_f1": "{:.2%}",
    "neutral_pr_auc": "{:.3f}",
    "false_emotional_rate_on_neutral_only": "{:.2%}",
    "false_neutral_rate_on_non_neutral": "{:.2%}",
}))

## 10. 每标签 F1 图

In [ ]:
plot_df = per_label_df.sort_values("f1", ascending=True)

fig, ax = plt.subplots(figsize=(10, 10))
y = np.arange(len(plot_df))
ax.barh(y, plot_df["f1"])
ax.set_yticks(y)
ax.set_yticklabels(plot_df["emotion"])
ax.set_xlim(0, 1)
ax.set_xlabel("F1")
ax.set_title(f"GoEmotions per-label F1: {REPORT_STRATEGY}")
ax.grid(axis="x", alpha=0.3)

for index, value in enumerate(plot_df["f1"]):
    ax.text(min(value + 0.015, 0.98), index, f"{value:.2f}", va="center", fontsize=8)

plt.tight_layout()
plt.savefig("experiment_c_per_label_f1.png", dpi=200, bbox_inches="tight")
plt.show()

## 11. 样本量与 F1 的关系

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(per_label_df["support"], per_label_df["f1"])

for _, row in per_label_df.iterrows():
    ax.annotate(
        row["emotion"],
        (row["support"], row["f1"]),
        xytext=(4, 3),
        textcoords="offset points",
        fontsize=7,
    )

ax.set_xscale("log")
ax.set_xlabel("Test support (log scale)")
ax.set_ylabel("F1")
ax.set_title("GoEmotions: label support vs F1")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("experiment_c_support_vs_f1.png", dpi=200, bbox_inches="tight")
plt.show()

## 12. PR-AUC 图

In [ ]:
plot_df = per_label_df.sort_values("pr_auc", ascending=True)

fig, ax = plt.subplots(figsize=(10, 10))
y = np.arange(len(plot_df))
ax.barh(y, plot_df["pr_auc"])
ax.set_yticks(y)
ax.set_yticklabels(plot_df["emotion"])
ax.set_xlim(0, 1)
ax.set_xlabel("Average Precision / PR-AUC")
ax.set_title("GoEmotions per-label PR-AUC")
ax.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.savefig("experiment_c_per_label_pr_auc.png", dpi=200, bbox_inches="tight")
plt.show()

## 13. 高置信度 False Positive 与 False Negative

- False Positive：模型很确定某情绪存在，但真实标签中没有。
- False Negative：真实标签存在，但模型给出的概率明显低于所选阈值。

In [ ]:
test_texts = np.asarray(test_ds["text"])

false_positive_records = []
false_negative_records = []

for label_index, label in enumerate(dataset_label_names):
    threshold = report_thresholds[label_index]
    true_col = y_test_true[:, label_index]
    pred_col = y_test_pred[:, label_index]
    prob_col = test_prob[:, label_index]

    fp_indices = np.where((true_col == 0) & (pred_col == 1))[0]
    fn_indices = np.where((true_col == 1) & (pred_col == 0))[0]

    for sample_index in fp_indices:
        false_positive_records.append({
            "emotion": label,
            "text": test_texts[sample_index],
            "probability": prob_col[sample_index],
            "threshold": threshold,
            "true_labels": ", ".join(
                np.asarray(dataset_label_names)[
                    y_test_true[sample_index].astype(bool)
                ]
            ),
        })

    for sample_index in fn_indices:
        false_negative_records.append({
            "emotion": label,
            "text": test_texts[sample_index],
            "probability": prob_col[sample_index],
            "threshold": threshold,
            "true_labels": ", ".join(
                np.asarray(dataset_label_names)[
                    y_test_true[sample_index].astype(bool)
                ]
            ),
        })

false_positive_df = (
    pd.DataFrame(false_positive_records)
    .sort_values("probability", ascending=False)
)
false_negative_df = (
    pd.DataFrame(false_negative_records)
    .sort_values("probability", ascending=True)
)

print("最高置信度 False Positive：")
display(false_positive_df.head(40))

print("最低概率 False Negative：")
display(false_negative_df.head(40))

## 14. 查看指定标签的错误样本

In [ ]:
LABEL_TO_INSPECT = "grief"
N_EXAMPLES = 10

print(f"标签：{LABEL_TO_INSPECT}")

print("False Positive：")
display(
    false_positive_df[
        false_positive_df["emotion"] == LABEL_TO_INSPECT
    ].head(N_EXAMPLES)
)

print("False Negative：")
display(
    false_negative_df[
        false_negative_df["emotion"] == LABEL_TO_INSPECT
    ].head(N_EXAMPLES)
)

## 15. 导出结果

In [ ]:
strategy_summary_df.to_csv(
    "experiment_c_threshold_strategy_summary.csv", index=False
)
global_threshold_search_df.to_csv(
    "experiment_c_global_threshold_search.csv", index=False
)
threshold_table.to_csv(
    "experiment_c_per_label_thresholds.csv", index=False
)
per_label_df.to_csv(
    "experiment_c_per_label_metrics.csv", index=False
)
neutral_summary_df.to_csv(
    "experiment_c_neutral_metrics.csv", index=False
)
false_positive_df.to_csv(
    "experiment_c_false_positives.csv", index=False
)
false_negative_df.to_csv(
    "experiment_c_false_negatives.csv", index=False
)

prediction_export = pd.DataFrame({
    "text": test_ds["text"],
    "true_labels": [
        ", ".join(
            np.asarray(dataset_label_names)[row.astype(bool)]
        )
        for row in y_test_true
    ],
    "predicted_labels": [
        ", ".join(
            np.asarray(dataset_label_names)[row.astype(bool)]
        )
        for row in y_test_pred
    ],
})
prediction_export.to_csv(
    "experiment_c_all_predictions.csv", index=False
)

print("实验 C 的 CSV 与 PNG 文件已保存到当前 Colab 运行目录。")

## 报告文字模板

> 本实验使用 `SamLowe/roberta-base-go_emotions` 在 GoEmotions 官方测试集上评估 28 类多标签情绪识别性能。为避免测试集调参，我们在验证集上分别选择全局阈值和逐标签阈值，并将其固定用于测试集。采用 `{阈值策略}` 时，模型的 Micro-F1、Macro-F1、Weighted-F1 和 Exact-match Accuracy 分别为 `{结果}`、`{结果}`、`{结果}` 和 `{结果}`。不同标签的 F1 存在明显差异，表现最佳的标签为 `{结果}`，表现最弱的标签为 `{结果}`，相差 `{结果}` 个百分点。标签测试样本量与 F1 的 Spearman 相关系数为 `{结果}`，说明 `{解释}`。Neutral 的 Precision、Recall 和 F1 分别为 `{结果}`、`{结果}` 和 `{结果}`。由于模型在同一数据集上训练，本结果主要反映模型在同分布测试集中的类别差异，而不是跨领域泛化能力。